In [2]:
import pandas as pd
from cand_gen.embedding import train_model
from cand_gen import triple_gen

/var/folders/_3/wtwzgv1d3rlfz233qkf36kg00000gp/T/ipykernel_39076/1236390600.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
/opt/homebrew/Caskroom/miniconda/base/envs/kg-emb/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('data/freebase/data.csv')
df = df.dropna()
candidates_df = triple_gen.generate_all_candidates(df.sample(1000))

In [ ]:
sub_df = df.sample(int(1*len(df)))
train_df = train_model.create_dataset(
    sub_df)
test_df = train_model.create_dataset(
    sub_df.sample(n=50))

model_name = "TransE"
embedding_dim = 5


model_kwargs = {"embedding_dim": embedding_dim}
experiment_name = model_name+f"_dim{embedding_dim}"
model = train_model.create_pipeline(train_df, test_df,
                    model_name, model_kwargs, experiment_name)

In [8]:
import pykeen
import numpy as np
from pykeen.pipeline import pipeline
from pykeen.models import TransE
from pykeen.triples import TriplesFactory

def get_triplet_emb(model: pykeen.models.TransE, triple_factory: pykeen.triples.TriplesFactory,
                    head: str, relation: str, tail: str) -> (np.ndarray, np.ndarray, np.ndarray):
    """ Return the triplet embedding from model, dataset and triplet list """
    # get total entity embedding
    entities_embedding = model.entity_representations[0](
        indices=None).detach().numpy()
    # get total relation embedding
    relations_embedding = model.relation_representations[0](
        indices=None).detach().numpy()

    # get id of each element of the triplet
    head_id = triple_factory.entity_to_id[head]
    relation_id = triple_factory.relation_to_id[relation]
    tail_id = triple_factory.entity_to_id[tail]

    # get every element embedding
    head_emb = entities_embedding[head_id]
    relation_emb = relations_embedding[relation_id]
    tail_emb = entities_embedding[tail_id]
    return (head_emb, relation_emb, tail_emb)


def compute_dist_emb(head_emb: np.ndarray, relation_emb: np.ndarray, tail_emb: np.ndarray) -> np.float32:
    """ Return computed distance of embedding """
    # sum of head and relation
    sum_head_relation = head_emb + relation_emb
    # difference between sum of head and relation and tail
    distance = np.linalg.norm(sum_head_relation - tail_emb)
    return distance


def get_list_dist(df: pd.DataFrame, model: pykeen.models.TransE, triple_factory: pykeen.triples.TriplesFactory,) -> list[np.float32]:
    """ Return list of distance for every embedded triple """
    # list of distance
    list_dist = []
    # iterate over the DataFrame
    for index, row in df.iterrows():
        head = row['Head']
        relation = row['Relation']
        tail = row['Tail']
        # get embedding
        head_emb, rel_emb, tail_emb = get_triplet_emb(
            model, triple_factory, head, relation, tail)
        # get distance of every triplet
        dist = compute_dist_emb(head_emb, rel_emb, tail_emb)
        list_dist.append(dist)
    return list_dist

In [12]:
from cand_gen import triple_gen
from cand_gen.embedding import train_model

#df = pd.read_csv('data/freebase/data.csv')
candidates_df = triple_gen.generate_all_candidates(df)

list_dist = get_list_dist(candidates_df, model.model, train_df)
candidates_df['distance'] = list_dist

KeyboardInterrupt: 

In [ ]:
from pykeen.pipeline import pipeline
from pykeen.models import TransE
from pykeen.triples import TriplesFactory

def get_triplet_emb(model: pykeen.models.TransE, triple_factory: pykeen.triples.TriplesFactory,
                    head: str, relation: str, tail: str) -> (np.ndarray, np.ndarray, np.ndarray):
    """ Return the triplet embedding from model, dataset and triplet list """
    # get total entity embedding
    entities_embedding = model.entity_representations[0](
        indices=None).detach().numpy()
    # get total relation embedding
    relations_embedding = model.relation_representations[0](
        indices=None).detach().numpy()

    # get id of each element of the triplet
    head_id = triple_factory.entity_to_id[head]
    relation_id = triple_factory.relation_to_id[relation]
    tail_id = triple_factory.entity_to_id[tail]

    # get every element embedding
    head_emb = entities_embedding[head_id]
    relation_emb = relations_embedding[relation_id]
    tail_emb = entities_embedding[tail_id]
    return (head_emb, relation_emb, tail_emb)


def compute_dist_emb(head_emb: np.ndarray, relation_emb: np.ndarray, tail_emb: np.ndarray) -> np.float32:
    """ Return computed distance of embedding """
    # sum of head and relation
    sum_head_relation = head_emb + relation_emb
    # difference between sum of head and relation and tail
    distance = np.linalg.norm(sum_head_relation - tail_emb)
    return distance


def get_list_dist(df: pd.DataFrame, model: pykeen.models.TransE, triple_factory: pykeen.triples.TriplesFactory,) -> list[np.float32]:
    """ Return list of distance for every embedded triple """
    # list of distance
    list_dist = []
    # iterate over the DataFrame
    for index, row in df.iterrows():
        head = row['Head']
        relation = row['Relation']
        tail = row['Tail']
        # get embedding
        head_emb, rel_emb, tail_emb = get_triplet_emb(
            model, triple_factory, head, relation, tail)
        # get distance of every triplet
        dist = compute_dist_emb(head_emb, rel_emb, tail_emb)
        list_dist.append(dist)
    return list_dist